In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl

graphs_dir = "results/graphs"
os.makedirs("../"+graphs_dir, exist_ok=True)
os.makedirs("../"+graphs_dir+"/means", exist_ok=True)

stats_dir = "results/stats"
os.makedirs("../"+stats_dir, exist_ok=True)

noises = [0] + [round(i*0.01, 2) for i in range(1, 11)]
tests = ['testAB', 'testAG', 'test']


In [2]:
methods_lst = ['grad01_avg', 'grad01_max', 'gradient_conf_avg', 
               'gradient_conf_max', 'gradient_input_avg', 'gradient_input_max', 
               'model_grad_avg', 
               'model_grad_max', 'qbc', 'threshold_avg', 
               'threshold_max', 'feature_eucl', 'random']

base_palette = ["#004949", "#009292", "#ff6db6", 
                "#ffb6db", "#490092", "#006ddb", 
                "#b6dbff", 
                "#920000", "#924900", "#db6d00", 
                "#24ff24", "#ffa500", "#ffff6d"]

user_palette = {methods_lst[i]: base_palette[i] for i in range(len(methods_lst))}
method_colors = dict(user_palette)

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

try:
    cmap = mpl.colormaps.get_cmap("tab20")
except AttributeError:
    cmap = plt.get_cmap("tab20")

if hasattr(cmap, "colors"):
    base_colors = list(cmap.colors)
else:
    N = getattr(cmap, "N", 20)
    base_colors = [cmap(i / max(N - 1, 1)) for i in range(N)]

all_methods = methods_lst
missing = [m for m in all_methods if m not in method_colors]
for i, method in enumerate(missing):
    rgba = base_colors[i % len(base_colors)]
    method_colors[method] = mcolors.to_hex(rgba)

method_labels = {m: m.replace('_', ' ') for m in methods_lst}


In [3]:
from tqdm import tqdm

df_runs = pd.DataFrame()
for noise in noises:
    for n in tqdm(range(100)):
        for m in methods_lst:
            filepath = f"../results_small_final/noise{noise}"+'/rand'+str(n)+'/roc_aucs_'+m+f'_{n}.tsv'
            try:
                df = pd.read_csv(filepath, sep='\t')
                
                df['noise'] = noise
                df['method'] = m
                df['run_id'] = n
                df_runs = pd.concat([df_runs, df])
            except:
                pass

df_runs2 = df_runs[['roc_aucs_test', 'roc_aucs_testAB',
           'roc_aucs_testAG', 'ags_number', 'run_id', 'noise', 'method']].drop_duplicates().reset_index(drop=True)

100%|█████████████████████████████████████████████████████████████| 100/100 [01:48<00:00,  1.08s/it]


In [6]:
df_runs2.to_csv('all.tsv', index=None, sep='\t')
df = pd.read_csv('all.tsv', sep='\t')

In [30]:
import numpy as np
import pandas as pd
from scipy import stats

tests = {
    "test": "roc_aucs_test",
    "testAB": "roc_aucs_testAB",
    "testAG": "roc_aucs_testAG"
}

def compute_auc_per_run_method(df, value_col, x_col='ags_number', run_col='run_id', method_col='method'):
    records = []
    for (run_id, method), sub in df.groupby([run_col, method_col]):
        sub2 = sub.dropna(subset=[x_col, value_col])
        if sub2.empty:
            auc = np.nan
        else:
            s = sub2.groupby(x_col)[value_col].mean().sort_index()
            if s.shape[0] >= 2:
                x = s.index.astype(float).to_numpy()
                y = s.values.astype(float)
                auc = np.trapezoid(y=y, x=x)
            else:
                auc = np.nan
        records.append({'run_id': str(run_id), 'method': method, 'auc': float(auc) if not np.isnan(auc) else np.nan})
    return pd.DataFrame.from_records(records)

def paired_tests_vs_random(aucs_long_df, alpha=0.05, alternative='two-sided'):
    results = []
    tests_list = sorted(aucs_long_df['test'].unique())
    for test in tests_list:
        sub = aucs_long_df[aucs_long_df['test'] == test]
        methods = sorted(sub['method'].unique())
        if 'random' not in methods:
            print(f"Test {test}: no 'random' method found — skipping")
            continue
        methods_to_test = [m for m in methods if m != 'random']
        m = len(methods_to_test)
        # print('m:', m)
        for method in methods_to_test:
            a = sub[sub['method'] == method][['run_id','auc']].set_index('run_id')
            b = sub[sub['method'] == 'random'][['run_id','auc']].set_index('run_id')
            paired = a.join(b, how='inner', lsuffix='_m', rsuffix='_r').dropna()
            n_pairs = paired.shape[0]
            if n_pairs < 2:
                results.append({
                    'test': test, 'method': method, 'n_pairs': n_pairs,
                    'mean_method': np.nan, 'mean_random': np.nan, 'mean_diff': np.nan,
                    't_stat': np.nan, 'p_val': np.nan, 'p_adj': np.nan, 'reject': False,
                    'cohen_d': np.nan, 'ci_lower_rel_pct': np.nan, 'ci_upper_rel_pct': np.nan
                })
                continue

            x = paired['auc_m'].to_numpy(dtype=float)
            y = paired['auc_r'].to_numpy(dtype=float)
            diff = x - y
            mean_method = x.mean()
            mean_random = y.mean()
            mean_diff = diff.mean()
            sd_diff = diff.std(ddof=1)

            try:
                t_res = stats.ttest_rel(x, y, alternative=alternative)
                t_stat = float(t_res.statistic)
                p_val = float(t_res.pvalue)
            except TypeError:
                # older scipy without 'alternative' argument
                t_stat, p_two = stats.ttest_rel(x, y)
                if alternative == 'two-sided':
                    p_val = float(p_two)
                elif alternative == 'greater':
                    p_val = float(p_two/2) if t_stat > 0 else 1.0 - float(p_two/2)
                elif alternative == 'less':
                    p_val = float(p_two/2) if t_stat < 0 else 1.0 - float(p_two/2)
                else:
                    p_val = float(p_two)

            cohen_d = (mean_diff / sd_diff) if sd_diff > 0 else np.nan

            # Confidence interval for the paired mean difference (two-sided, level = 1 - alpha)
            if n_pairs > 1:
                se_diff = sd_diff / np.sqrt(n_pairs)
                # two-sided critical t
                t_crit = stats.t.ppf(1 - alpha/2, df=n_pairs - 1)
                ci_lower = (mean_diff - t_crit * se_diff)/mean_random*100
                ci_upper = (mean_diff + t_crit * se_diff)/mean_random*100
            else:
                ci_lower = np.nan
                ci_upper = np.nan

            # Bonferroni correction across methods for this test
            p_adj = min(p_val * m, 1.0) if m > 0 else p_val
            reject = (p_adj < alpha)

            results.append({
                'test': test, 'method': method, 'n_pairs': n_pairs,
                'mean_method': mean_method, 'mean_random': mean_random, 'mean_diff': mean_diff,
                't_stat': t_stat, 'p_val': p_val, 'p_adj': p_adj, 'reject': bool(reject),
                'cohen_d': cohen_d, 'ci_lower_rel_pct': ci_lower, 'ci_upper_rel_pct': ci_upper
            })

    results_df = pd.DataFrame(results)
    cols = ['test','method','n_pairs','mean_method','mean_random','mean_diff',
            't_stat','p_val','p_adj','reject','ci_lower_rel_pct','ci_upper_rel_pct']
    results_df = results_df[[c for c in cols if c in results_df.columns]]
    return results_df


def f2(dfe):
    aucs_dfs = []
    for test_name, col in tests.items():
        
        if col not in dfe.columns:
            print(f"Warning: column {col} not found in dfe — skipping {test_name}")
            continue
        a = compute_auc_per_run_method(dfe, value_col=col, x_col='ags_number', run_col='run_id', method_col='method')
        a['test'] = test_name
        aucs_dfs.append(a)
    
    if not aucs_dfs:
        raise RuntimeError("No AUCs computed: none of the requested columns found in dfe.")
    
    aucs_long = pd.concat(aucs_dfs, ignore_index=True)
    ttest_results = paired_tests_vs_random(aucs_long, alpha=0.05, alternative='greater')

    return ttest_results


In [31]:
ttest_results = f2(df)

In [32]:
ttest_results['mean_diff_rel_pct'] = ttest_results.mean_diff/ttest_results.mean_random*100

ttest_results[ttest_results.reject==True]

,test,method,n_pairs,mean_method,mean_random,mean_diff,t_stat,p_val,p_adj,reject,ci_lower_rel_pct,ci_upper_rel_pct,mean_diff_rel_pct
9,test,qbc,100,67.873689,67.285587,0.588103,4.879784,2.033904e-06,2.440684e-05,True,0.518638,1.229441,0.874039
10,test,threshold_avg,100,68.157938,67.285587,0.872351,4.642701,5.287203e-06,6.344643e-05,True,0.742391,1.850590,1.296490
11,test,threshold_max,100,68.076963,67.285587,0.791377,4.395454,1.390854e-05,1.669025e-04,True,0.645204,1.707087,1.176146
21,testAB,qbc,100,72.803720,72.241658,0.562061,7.725835,4.617374e-12,5.540849e-11,True,0.578209,0.977849,0.778029
22,testAB,threshold_avg,100,72.908169,72.241658,0.666510,6.781493,4.369484e-10,5.243381e-09,True,0.652662,1.192562,0.922612
23,testAB,threshold_max,100,72.783581,72.241658,0.541922,5.440046,1.928115e-07,2.313739e-06,True,0.476540,1.023764,0.750152
33,testAG,qbc,100,73.695050,73.009476,0.685573,6.323121,3.715229e-09,4.458275e-08,True,0.644352,1.233687,0.939020
34,testAG,threshold_avg,100,73.713006,73.009476,0.703529,4.209550,2.819151e-05,3.382982e-04,True,0.509404,1.417824,0.963614
35,testAG,threshold_max,100,73.683023,73.009476,0.673547,4.377303,1.491380e-05,1.789656e-04,True,0.504360,1.340735,0.922547


In [33]:
out_tsv = os.path.join("..", stats_dir, "stats.tsv")
ttest_results.to_csv(out_tsv, index=None, sep='\t')
